In [2]:
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook

file_path = Path("当周金额合计.xlsx")
header_row = 1  # 改成这个文件的实际表头行号

workbook = load_workbook(file_path, read_only=True, data_only=True)
try:
    sheet = workbook.worksheets[0]
    rows = sheet.iter_rows(min_row=header_row, values_only=True)
    headers = [str(value).strip() for value in next(rows)]
    market_idx = headers.index("发卡分公司")
    amount_idx = headers.index("当周金额合计")

    samples = {"中国大陆": [], "非中国大陆": []}
    for row in rows:
        market = row[market_idx]
        amount = row[amount_idx]

        # 对齐 Excel 的 ISNUMBER：排除空白、文本数字和布尔值。
        if market in (None, "") or type(amount) not in (int, float):
            continue

        group = "中国大陆" if market == "中国大陆" else "非中国大陆"
        samples[group].append(amount)
finally:
    workbook.close()

percentiles = [0.50, 0.75, 0.90, 0.99]
result = []

for group, amounts in samples.items():
    values = pd.Series(amounts, dtype="float64")
    thresholds = values.quantile(percentiles, interpolation="linear")
    result.append({
        "市场组": group,
        "样本数": len(values),
        "零金额数": int(values.eq(0).sum()),
        **{f"P{round(p * 100)}": thresholds.loc[p] for p in percentiles},
    })

print(pd.DataFrame(result).to_string(index=False))

  市场组  样本数  零金额数        P50         P75         P90          P99
 中国大陆 1044   433    207.960   23443.505  405589.008 3.043493e+07
非中国大陆  372    69 174155.495 1654254.800 8225550.430 1.487123e+08


In [7]:
import sqlite3
from pathlib import Path

import pandas as pd

db_path = Path("weekly_test.db").resolve()
week_dt = "2026-11-23"

with sqlite3.connect(f"{db_path.as_uri()}?mode=ro", uri=True) as conn:
    df = pd.read_sql_query(
        """
        SELECT issuer_branch, current_week_txn_amount
        FROM issuer_weekly_monitor_data;
        """,
        conn,
    )

df["市场组"] = df["issuer_branch"].eq("中国大陆").map({
    True: "中国大陆",
    False: "非中国大陆",
})

amounts = df.groupby("市场组")["current_week_txn_amount"]
result = amounts.quantile(
    [0.50, 0.75, 0.90, 0.99], interpolation="linear"
).unstack()
result.columns = ["P50", "P75", "P90", "P99"]
result.insert(0, "样本数", amounts.size())
result.insert(1, "零金额数", df["current_week_txn_amount"].eq(0).groupby(df["市场组"]).sum())

print(result.to_string())

        样本数  零金额数        P50         P75          P90           P99
市场组                                                                
中国大陆   1042   216    1518.61    63716.81   788218.060  3.877873e+07
非中国大陆   372    40  301240.92  2071594.49  8987869.996  1.608818e+08


In [10]:
import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd

root = Path("/Users/meixiaohan/Desktop/UnionpayInter/数据库")
db_path = root / "weekly_test.db"
excel_path = root / "当周金额合计.xlsx"  # 文件名不同的话改这里
output_path = root / "当周金额对账.xlsx"
week_dt = "2026-11-23"

key_map = {
    "发卡分公司": "issuer_branch",
    "发卡市场": "issuer_market",
    "发卡机构名称": "issuer_name",
    "发卡机构代码": "issuer_code",
}
join_keys = list(key_map.values())

with closing(sqlite3.connect(f"{db_path.as_uri()}?mode=ro", uri=True)) as conn:
    db_df = pd.read_sql_query(
        """
        SELECT *
        FROM issuer_weekly_monitor_data
        WHERE week_dt = ?
        """,
        conn,
        params=(week_dt,),
    )

excel_df = pd.read_excel(
    excel_path,
    dtype={"发卡机构代码": "string"},
    keep_default_na=False,
).rename(columns=key_map)  # 只重命名四个连接字段

for frame in (db_df, excel_df):
    for key in join_keys:
        frame[key] = frame[key].astype("string").fillna("").str.strip()

result = db_df.merge(
    excel_df[join_keys + ["当周金额合计"]],
    on=join_keys,
    how="outer",
    indicator=True,
    validate="one_to_one",
)

result["匹配状态"] = result.pop("_merge").map({
    "both": "已匹配",
    "left_only": "仅 SQLite 有",
    "right_only": "仅 Excel 有",
})

excel_amount = pd.to_numeric(result["当周金额合计"], errors="coerce")
result["金额差_数据库减Excel"] = (
    result["current_week_txn_amount"] - excel_amount
).round(2)

result.to_excel(output_path, index=False, engine="openpyxl")